# Stabilizer state → fault-tolerant trivalent ZX diagram

This notebook demonstrates every stage of the `spiderstate` stabilizer-to-unidealized pipeline:

1. validate a signed pure stabilizer state;
2. find and optimize an LC-equivalent graph state;
3. construct the ideal graph-state ZX diagram;
4. apply Lemma B* once to every original graph edge;
5. replace every remaining high-arity spider with an exactly verified SpiderCat gadget; and
6. verify that all three diagram stages prepare the exact input state, up to a nonzero global scalar.

The main example is the logical zero state of the five-qubit code. The construction follows the fault semantics of [Fault Tolerance by Construction](https://arxiv.org/abs/2506.17181), the trivalent gadgets of [SpiderCat](https://arxiv.org/abs/2603.05391), and the CSSCat staging generalized to arbitrary pure stabilizer states.

## Conventions

- `t` is the **inclusive** number of internal-edge faults to tolerate; supported values are `1..7`.
- The local-Clifford certificate uses `U |psi> = |G>`.
- The inverse local Cliffords are kept as output-boundary boxes, so the diagrams prepare the exact input state rather than only an LC representative.
- Blue edges are Hadamard edges; black edges are simple edges. Dashed edges are ideal and solid edges are noisy.
- ZX diagrams are compared up to nonzero global scalar.

Run `uv sync --python python3.12` from the repository root and select the resulting `.venv` kernel before executing the notebook.

In [ ]:
from collections import Counter

import networkx as nx
import numpy as np
import stim
from IPython.display import SVG, display

from spiderstate import (
    LCSearchConfig,
    SynthesisStage,
    UnsupportedFaultToleranceError,
    css_logical_state_stabilizers,
    stabilizer_state_to_graph,
    synthesize_stabilizer_state,
)
from spiderstate.spidercat_gadgets import (
    decompose_spidercats,
    predicted_spidercat_spider_count,
    verified_spidercat_gadget,
    verify_t_robustness,
)
from spiderstate.zx_ir import (
    EdgeKind,
    EdgeRole,
    FaultStatus,
    NodeKind,
    apply_lemma_b_star,
    build_ideal_graph_state_diagram,
)


def diagram_summary(diagram):
    """Return the main structural invariants of an annotated ZX diagram."""
    spiders = [
        node
        for node, data in diagram.graph.nodes(data=True)
        if data["kind"] in (NodeKind.Z_SPIDER, NodeKind.X_SPIDER)
    ]
    internal_edges = diagram.internal_edges()
    return {
        "stage": diagram.metadata.get("stage"),
        "total_nodes": diagram.graph.number_of_nodes(),
        "spider_kinds": dict(Counter(
            diagram.graph.nodes[node]["kind"].value for node in spiders
        )),
        "max_spider_arity": max(
            (diagram.graph.degree(node) for node in spiders), default=0
        ),
        "internal_edge_kinds": dict(Counter(
            data["kind"].value for _, _, data in internal_edges
        )),
        "ideal_internal_edges": sum(
            data["fault_status"] is FaultStatus.IDEAL
            for _, _, data in internal_edges
        ),
    }


def show_diagram(diagram):
    """Display the deterministic project-owned SVG renderer inline."""
    display(SVG(data=diagram.render_svg()))


def compare_up_to_scalar(actual, expected):
    """Return the fitted scalar and maximum residual."""
    actual = np.asarray(actual).reshape(-1)
    expected = np.asarray(expected).reshape(-1)
    support = np.flatnonzero(np.abs(expected) > 1e-8)
    if not len(support):
        raise ValueError("Expected state vector is zero.")
    scalar = actual[support[0]] / expected[support[0]]
    residual = float(np.max(np.abs(actual - scalar * expected)))
    return scalar, residual

## 1. Specify and validate the stabilizer state

The first four generators below are the five-qubit code stabilizers. `ZZZZZ` fixes the logical-Z eigenvalue to `+1`, selecting logical zero.

In [ ]:
five_qubit_logical_zero = (
    "+XZZXI",
    "+IXZZX",
    "+XIXZZ",
    "+ZXIXZ",
    "+ZZZZZ",
)
t = 1

# Stim independently validates commutation, independence, signs, and purity.
input_tableau = stim.Tableau.from_stabilizers(
    [stim.PauliString(generator) for generator in five_qubit_logical_zero]
)
print("Canonical input stabilizers:")
for generator in input_tableau.to_stabilizers(canonicalize=True):
    print(" ", generator)

## 2. Convert to and optimize an LC-equivalent graph state

`stabilizer_state_to_graph` performs the signed binary-symplectic reduction, constructs the local-Clifford certificate, and searches the local-complementation orbit. Through eight qubits the default search exhausts the orbit. The arity-cost callback below makes the low-level call use the same `t`-dependent SpiderCat cost model as the one-call pipeline.

In [ ]:
lc_search = LCSearchConfig(
    vertex_arity_cost=lambda arity: predicted_spidercat_spider_count(arity, t)
)
graph_result = stabilizer_state_to_graph(
    five_qubit_logical_zero,
    lc_search=lc_search,
)
G = graph_result.graph

print("Graph nodes:", list(G.nodes()))
print("Graph edges:", sorted(G.edges()))
print("Degrees:", dict(G.degree()))
print("LC search:", graph_result.search)
print("Isomorphic to C5:", nx.is_isomorphic(G, nx.cycle_graph(5)))
print("Certificate valid:", graph_result.validate_certificate())

assert nx.is_isomorphic(G, nx.cycle_graph(5))
assert graph_result.search.guaranteed_optimal

print("\nPer-qubit local Cliffords:")
for qubit, (to_graph, from_graph) in enumerate(zip(
    graph_result.local_cliffords_to_graph,
    graph_result.local_cliffords_from_graph,
    strict=True,
)):
    print(
        f" q{qubit}: input→graph {to_graph.gate_word}; "
        f"graph→input {from_graph.gate_word}"
    )

## 3. Build the ideal graph-state ZX diagram

Every graph vertex becomes a phase-zero Z spider with one output. Every graph edge becomes an **ideal Hadamard edge**. The purple output boxes hold the inverse local Cliffords and therefore recover the exact input state.

In [ ]:
ideal_diagram = build_ideal_graph_state_diagram(
    G,
    local_corrections=graph_result.local_cliffords_from_graph,
)

print(diagram_summary(ideal_diagram))
print("Original graph-edge records:")
for _, _, data in ideal_diagram.edges_of_role(EdgeRole.GRAPH_EDGE):
    print(
        f" {data['edge_id']}: {data['kind'].value}, "
        f"{data['fault_status'].value}"
    )

assert len(ideal_diagram.edges_of_role(EdgeRole.GRAPH_EDGE)) == 5
assert all(
    data["kind"] is EdgeKind.HADAMARD
    and data["fault_status"] is FaultStatus.IDEAL
    for _, _, data in ideal_diagram.edges_of_role(EdgeRole.GRAPH_EDGE)
)
show_diagram(ideal_diagram)

## 4. Unidealize every original edge with Lemma B*

For each original ideal Hadamard edge `u-v`, the rewrite introduces two phase-zero X spiders on each side. The four X spiders form a noisy Hadamard `K2,2`; each endpoint connects to its two local X spiders by noisy simple edges. Newly introduced Hadamard edges are not rewritten recursively.

In [ ]:
post_lemma_b_star = apply_lemma_b_star(ideal_diagram)
post_summary = diagram_summary(post_lemma_b_star)
print(post_summary)

lemma_x_spiders = post_lemma_b_star.nodes_of_kind(NodeKind.X_SPIDER)
graph_z_spiders = post_lemma_b_star.nodes_of_kind(NodeKind.Z_SPIDER)

print("Lemma-B* X spiders:", len(lemma_x_spiders))
print("Original Z-spider arities:", [
    post_lemma_b_star.spider_arity(node) for node in graph_z_spiders
])

assert len(lemma_x_spiders) == 20
assert all(post_lemma_b_star.spider_arity(node) == 3 for node in lemma_x_spiders)
assert all(post_lemma_b_star.spider_arity(node) == 5 for node in graph_z_spiders)
assert not post_lemma_b_star.edges_of_role(EdgeRole.GRAPH_EDGE)
assert all(
    data["fault_status"] is FaultStatus.NOISY
    for _, _, data in post_lemma_b_star.internal_edges()
)

# Applying the method again is an idempotent copy, not a recursive rewrite.
assert apply_lemma_b_star(post_lemma_b_star).to_json() == post_lemma_b_star.to_json()
show_diagram(post_lemma_b_star)

## 5. Decompose high-arity spiders with SpiderCat

The introduced X spiders are already trivalent. Each arity-five Z spider is replaced by a verified pentagon of five phase-zero Z spiders. For larger arities the provider loads or searches a marked graph and then independently checks the exact marked-cut condition using the actual cut size `f`.

First inspect the verified gadget selected for one arity-five spider. Its five marked vertices are the stable external attachment ports.

In [ ]:
arity_five_gadget = verified_spidercat_gadget(arity=5, t=t)

print("Construction:", arity_five_gadget.construction)
print("Requested/effective t:", (
    arity_five_gadget.requested_t,
    arity_five_gadget.effective_t,
))
print("Optimality label:", arity_five_gadget.optimality)
print("Attachment nodes:", arity_five_gadget.attachment_nodes)
print("Gadget edges:", sorted(arity_five_gadget.graph.edges()))

assert arity_five_gadget.construction == "verified-cycle-5"
assert len(arity_five_gadget.attachment_nodes) == 5
assert verify_t_robustness(arity_five_gadget.graph, t=t)


In [ ]:
final_diagram, spidercat_metadata = decompose_spidercats(
    post_lemma_b_star,
    t=t,
)
final_summary = diagram_summary(final_diagram)
print(final_summary)

replacement_summary = [
    {
        "source": replacement.source_node,
        "arity": replacement.arity,
        "construction": replacement.construction,
        "effective_t": replacement.effective_t,
        "gadget_spiders": len(replacement.gadget_nodes),
        "ports": len(replacement.ports),
    }
    for replacement in spidercat_metadata.replacements
]
replacement_summary

In [ ]:
# Extract the spider-only graph requested by the synthesis problem.
spider_nodes = [
    node
    for node, data in final_diagram.graph.nodes(data=True)
    if data["kind"] in (NodeKind.Z_SPIDER, NodeKind.X_SPIDER)
]
trivalent_core = final_diagram.graph.subgraph(spider_nodes).copy()

node_kind_counts = Counter(
    final_diagram.graph.nodes[node]["kind"].value for node in spider_nodes
)
edge_kind_counts = Counter(
    data["kind"].value for _, _, data in final_diagram.internal_edges()
)

print("Spider-only graph nodes:", trivalent_core.number_of_nodes())
print("Spider kinds:", dict(node_kind_counts))
print("Internal edge kinds:", dict(edge_kind_counts))
maximum_spider_arity = max(
    final_diagram.graph.degree(node) for node in spider_nodes
)
print("Maximum spider arity (including output legs):", maximum_spider_arity)
print("All internal edges noisy:", all(
    data["fault_status"] is FaultStatus.NOISY
    for _, _, data in final_diagram.internal_edges()
))

assert node_kind_counts[NodeKind.Z_SPIDER.value] == 25
assert node_kind_counts[NodeKind.X_SPIDER.value] == 20
assert trivalent_core.number_of_nodes() == 45
assert maximum_spider_arity <= 3
assert all(
    data["fault_status"] is FaultStatus.NOISY
    for _, _, data in final_diagram.internal_edges()
)
assert final_diagram.metadata["stage"] == "unidealized_trivalent"

show_diagram(final_diagram)

## 6. Inspect stable source-to-gadget port provenance

Every final attachment records the source spider, original neighbor and edge, preserved edge semantics, and final attachment node.

In [ ]:
first_replacement = spidercat_metadata.replacements[0]
print("Source spider:", first_replacement.source_node)
print("Source provenance:", first_replacement.source_provenance)
print("Final ports:")
for port in first_replacement.ports:
    print(
        f" port {port.port_index}: edge={port.original_edge_id}, "
        f"neighbor={port.original_neighbor}, "
        f"kind={port.original_edge_kind.value}, "
        f"attachment={port.attachment_node}"
    )

## 7. Run the same pipeline through the public one-call API

The staged calls above are useful for research and inspection. Normal callers can obtain all artifacts and metadata from one result object.

In [ ]:
result = synthesize_stabilizer_state(
    five_qubit_logical_zero,
    t=t,
)

print("Optimized graph edges:", sorted(result.graph.edges()))
print("Search metadata:", result.search_metadata)
print("Guarantee metadata:", result.guarantee_metadata)
print("SpiderCat replacements:", len(result.gadget_metadata.replacements))

assert nx.is_isomorphic(result.graph, nx.cycle_graph(5))
assert result.graph_conversion.validate_certificate()
assert result.final_diagram.to_json() == final_diagram.to_json()

## 8. Verify exact state semantics at every stage

The project-owned IR converts to PyZX. We compare each tensor with Stim's state vector for the original signed stabilizers. A nonzero fitted scalar is ignored, as in ZX semantics.

In [ ]:
expected_state = input_tableau.to_state_vector(endian="big")

for stage in SynthesisStage:
    actual_tensor = result.to_pyzx(stage=stage).to_tensor()
    scalar, residual = compare_up_to_scalar(actual_tensor, expected_state)
    print(
        f"{stage.value:>12}: scalar={scalar:.6g}, "
        f"max residual={residual:.3e}"
    )
    assert abs(scalar) > 1e-10
    assert residual < 1e-8

## 9. CSS/QECC helper and Stim Tableau adapter

For a CSS code, complete the code checks with signed logical-X or logical-Z eigenvalue generators. The resulting strings—or an equivalent `stim.Tableau`—can be passed directly to the same pipeline.

In [ ]:
# ZZI, IZZ, and logical XXX specify the three-qubit logical-plus/GHZ state.
css_plus_generators = css_logical_state_stabilizers(
    h_x=[],
    h_z=["110", "011"],
    logical_x=["111"],
    logical_z=["100"],
    state="+",
)
print("Completed CSS generators:", css_plus_generators)

css_tableau = stim.Tableau.from_stabilizers([
    stim.PauliString(generator) for generator in css_plus_generators
])
css_result = synthesize_stabilizer_state(css_tableau, t=1)

print("CSS graph edges:", sorted(css_result.graph.edges()))
print("CSS final summary:", diagram_summary(css_result.final_diagram))
assert css_result.graph_conversion.validate_certificate()
assert max(
    css_result.final_diagram.graph.degree(node)
    for node, data in css_result.final_diagram.graph.nodes(data=True)
    if data["kind"] in (NodeKind.Z_SPIDER, NodeKind.X_SPIDER)
) <= 3

## 10. Guarantee-preserving failure behavior

Unsupported fault levels and unavailable or failed exact gadget checks raise typed errors. The implementation never silently substitutes a weaker guarantee.

In [ ]:
try:
    synthesize_stabilizer_state(["+Z"], t=8)
except UnsupportedFaultToleranceError as error:
    print(type(error).__name__ + ":", error)

## 11. Reusable template

Replace the signed generators and `t` below with any complete pure stabilizer state.

In [ ]:
def synthesize_and_show(stabilizers, *, t):
    synthesized = synthesize_stabilizer_state(stabilizers, t=t)
    print("Graph edges:", sorted(synthesized.graph.edges()))
    print("Final summary:", diagram_summary(synthesized.final_diagram))
    show_diagram(synthesized.final_diagram)
    return synthesized


# Example: the Bell state stabilized by +XX and +ZZ.
bell_result = synthesize_and_show(("+XX", "+ZZ"), t=1)

## Result

Starting only from signed stabilizer generators, the pipeline returned:

- an optimized LC-equivalent graph and exact local-Clifford certificate;
- the ideal graph-state diagram;
- the one-time Lemma-B* expansion;
- an exactly verified noisy graph containing only phase-zero Z/X spiders of arity at most three and simple/Hadamard edges; and
- deterministic SVG/PyZX adapters plus complete source-to-port provenance.

For the five-qubit logical-zero state, the selected graph is `C5`, Lemma B* introduces 20 X spiders, and SpiderCat produces 25 Z spiders, for 45 trivalent spiders in total.